In [1]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

In [2]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
#warnings.filterwarnings('ignore')

In [3]:
# Time
start = dt.datetime(2019,4,19)
end = dt.datetime(2019,4,24)
print(start,end,end-start)

2019-04-19 00:00:00 2019-04-24 00:00:00 5 days, 0:00:00


In [4]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
con_cursor = cursor.superstars.matches
aw = []
for documents in con_cursor.find({'created_at': {'$lt': end, '$gte': start}},
                                 {"home_team":1, 'away_team':1, "winner_team":1,"status":1,"type":1,"start_time":1}):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
df = pd.DataFrame(dic_flattened)
matches = df.loc[:,['_id','away_team_id','away_team_name','winner_team_id','home_team_id','start_time','status','type']]
matches = matches[matches.type == 'CAMPAIGN']
matches.sort_values('start_time',inplace=True)
print(len(matches))
matches.drop_duplicates(['away_team_id','home_team_id'],inplace = True)
print(len(matches))

43731
31573


In [5]:
matches['complete'] = (matches.status == 3)
f = {'away_team_name':'first','complete':'sum','start_time':'count'}

#completed = matches.groupby('away_team_id')['complete'].sum()
df2 = matches.groupby('away_team_id').agg(f)
df2.rename(columns={'start_time':'total matches'}, inplace=True)
df2['completion rate'] = df2['complete']/df2['total matches']
df2.head(20)

,away_team_name,complete,total matches,completion rate
away_team_id,,,,
5c876f2a5b2cd56774774fe6,Kolkata Eagles,4644.0,6266,0.741143
5c876f2b5b2cd56774775003,Hyderabad Lions,3107.0,4246,0.731748
5c876f2b5b2cd56774775020,Chennai Raptors,1968.0,2665,0.738462
5c876f2b5b2cd5677477503d,Pune Tigers,1487.0,1802,0.825194
5c876f2b5b2cd5677477505a,Jaipur Rangers,1242.0,1492,0.832440
5c876f2b5b2cd56774775077,Ahmedabad Jets,1011.0,1249,0.809448
5c876f2b5b2cd56774775094,Bengaluru Dynamites,848.0,986,0.860041
5c876f2c5b2cd567747750b4,Bhopal Mavericks,687.0,814,0.843980
5c876f2c5b2cd567747750d4,Nagpur Heroes,646.0,724,0.892265


In [79]:
matches['win'] = (matches.home_team_id == matches.winner_team_id)
df3 = matches.groupby('away_team_id')['win'].sum()
df3 = df3.to_frame().reset_index()
df3.head()

,away_team_id,win
0,5c876f2a5b2cd56774774fe6,4644.0
1,5c876f2b5b2cd56774775003,2664.0
2,5c876f2b5b2cd56774775020,1576.0
3,5c876f2b5b2cd5677477503d,1293.0
4,5c876f2b5b2cd5677477505a,1102.0


In [1]:
A = pd.merge(df2,df3,on='away_team_id',how = 'outer')
A['win rate'] = A['win']/A['complete']
A.drop(['win'], axis=1,inplace = True)

A['completion rate'] = (A['completion rate']*100).round()
A['win rate'] = (A['win rate']*100).round()
#df.value1 = df.value1.round()
#A.sort_values('win rate',inplace = True,ascending = False)
print(A.head())

NameError: name 'pd' is not defined